# ECG Training Notebook — DAT255

End-to-end training pipeline for the PTB-XL ECG classification project.
This notebook is responsible for everything that produces a trained model;
the `comparison.ipynb` notebook only consumes the artifacts it saves.

**Pipeline:**
1. Setup & reproducibility
2. Load dataset + build multi-hot labels
3. Train/val/test split (PTB-XL folds), normalisation, class weights
4. Data augmentation (time shift, Gaussian noise, amplitude scaling)
5. Model definitions — CNN, Residual CNN, BiLSTM
6. Focal loss
7. **Keras Tuner hyperparameter search on CNN**
8. Final training runs (CNN, ResCNN, BiLSTM) with MLflow
9. Save everything the comparison notebook will need

**Artifacts produced** (in `artifacts/`):
- `ecg_cnn_final.keras`, `ecg_rescnn_final.keras`, `ecg_lstm_final.keras`
- `normalisation_params.npz`
- `eval_sets.npz` (X_val, y_val, X_test, y_test — for comparison notebook)
- `histories.pkl` (training curves for all models)
- `tuner_results.json` (Keras Tuner summary)

## 1. Setup

In [ ]:
# Colab setup — run once
# !pip install -q wfdb keras tensorflow mlflow keras-tuner

In [ ]:
import os, sys, json, pickle, random, zipfile, ast
os.environ.pop("MLFLOW_TRACKING_URI", None)
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
import tensorflow as tf
import keras
from keras import layers, callbacks, optimizers
import keras_tuner as kt
from sklearn.preprocessing import MultiLabelBinarizer

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# Artifact directory
ART_DIR = "artifacts"
os.makedirs(ART_DIR, exist_ok=True)

print(f"Keras   : {keras.__version__}")
print(f"TF      : {tf.__version__}")
print(f"Tuner   : {kt.__version__}")
print(f"Backend : {keras.backend.backend()}")

## 2. Load dataset & build labels

In [ ]:
# Paths — adjust for your environment
ZIP_PATH = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
DATA_DIR = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

if not os.path.isdir(DATA_DIR):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Done.")

# Load metadata
df = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"), index_col="ecg_id")
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

# SCP -> superclass mapping
scp_df = pd.read_csv(os.path.join(DATA_DIR, "scp_statements.csv"), index_col=0)
scp_df = scp_df[scp_df["diagnostic"] == 1]
code_to_superclass = scp_df["diagnostic_class"].to_dict()

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(SUPERCLASSES)

def extract_superclasses(scp_dict):
    return list({code_to_superclass[c] for c in scp_dict
                 if c in code_to_superclass})

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].map(len) > 0].copy()   # drop unlabelled
df = df[df["age"] <= 120].copy()                  # drop age outliers

mlb = MultiLabelBinarizer(classes=SUPERCLASSES)
labels = mlb.fit_transform(df["superclasses"])
df[SUPERCLASSES] = labels

print(f"Samples after filtering: {len(df)}")
print("Class distribution:")
for i, sc in enumerate(SUPERCLASSES):
    n = labels[:, i].sum()
    print(f"  {sc:5s}: {n:5d}  ({n/len(df)*100:5.1f}%)")

In [ ]:
# Load signals (100 Hz, 1000 samples, 12 leads)
SAMPLING_RATE = 100
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF",
              "V1", "V2", "V3", "V4", "V5", "V6"]

def load_raw_signals(df, data_dir):
    col = "filename_lr"   # 100 Hz version
    signals = []
    for _, row in df.iterrows():
        rec = wfdb.rdrecord(os.path.join(data_dir, row[col]))
        signals.append(rec.p_signal)
    return np.array(signals, dtype=np.float32)

print("Loading ECG signals (takes a few minutes)...")
X_all = load_raw_signals(df, DATA_DIR)
y_all = labels.astype(np.float32)
print(f"X_all: {X_all.shape}   y_all: {y_all.shape}")

## 3. Split, normalise, class weights

In [ ]:
# PTB-XL official split: folds 1-8 train, 9 val, 10 test
folds = df["strat_fold"].values
train_mask = folds <= 8
val_mask   = folds == 9
test_mask  = folds == 10

X_train_raw, y_train = X_all[train_mask], y_all[train_mask]
X_val_raw,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test_raw,  y_test  = X_all[test_mask],  y_all[test_mask]

# Per-channel z-score (fit on train only)
train_mean = X_train_raw.mean(axis=(0, 1), keepdims=True)
train_std  = X_train_raw.std(axis=(0, 1), keepdims=True)
train_std[train_std == 0] = 1.0

def normalise(X):
    return np.clip((X - train_mean) / train_std, -10.0, 10.0)

X_train = normalise(X_train_raw)
X_val   = normalise(X_val_raw)
X_test  = normalise(X_test_raw)

# Save normalisation + test set for the comparison notebook
np.savez(os.path.join(ART_DIR, "normalisation_params.npz"),
         mean=train_mean, std=train_std)
np.savez(os.path.join(ART_DIR, "eval_sets.npz"),
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test)

print(f"Train: {X_train.shape}   Val: {X_val.shape}   Test: {X_test.shape}")

In [ ]:
# Class weights for multi-label BCE (neg/pos ratio per class)
pos = y_train.sum(axis=0)
neg = len(y_train) - pos
class_weights = neg / (pos + 1e-7)

print("Class weights (neg/pos ratio):")
for sc, w in zip(SUPERCLASSES, class_weights):
    print(f"  {sc:5s}: {w:.2f}")

## 4. Data augmentation

Three cheap, signal-preserving augmentations applied on the fly during training:
- **Random time shift** — rolls the signal by ±50 samples (±0.5s)
- **Gaussian noise** — σ=0.02 on normalised signal
- **Amplitude scaling** — random factor in [0.9, 1.1]

These simulate real-world variation (slightly different recording start times,
sensor noise, patient-to-patient amplitude differences) without changing the
diagnostic content. All are applied with 50% probability per-sample.

In [ ]:
def augment_ecg(x, y):
    """tf.data augmentation — x shape (T, 12), y shape (5,)."""
    # Random time shift (±50 samples)
    if tf.random.uniform(()) < 0.5:
        shift = tf.random.uniform((), -50, 51, dtype=tf.int32)
        x = tf.roll(x, shift, axis=0)

    # Gaussian noise
    if tf.random.uniform(()) < 0.5:
        x = x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.02)

    # Amplitude scaling
    if tf.random.uniform(()) < 0.5:
        scale = tf.random.uniform((), 0.9, 1.1)
        x = x * scale

    return x, y


def make_dataset(X, y, batch_size=64, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 2048), seed=SEED)
    if augment:
        ds = ds.map(augment_ecg, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


BATCH_SIZE = 64
train_ds = make_dataset(X_train, y_train, BATCH_SIZE, augment=True, shuffle=True)
val_ds   = make_dataset(X_val, y_val, BATCH_SIZE)
test_ds  = make_dataset(X_test, y_test, BATCH_SIZE)

## 5. Model definitions

Three architectures to compare:
- **CNN** — baseline 1D-CNN (4 conv blocks, GAP, dense head)
- **ResCNN** — residual variant with skip connections, deeper, better gradient flow
- **BiLSTM** — conv front-end + 2× bidirectional LSTM

In [ ]:
INPUT_SHAPE = (1000, 12)

def build_cnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
              filters=(32, 64, 128, 256), dropout=0.3, dense_units=128):
    """Baseline 1D-CNN. Used both for tuning and final training."""
    inputs = keras.Input(shape=input_shape, name="ecg_input")
    x = inputs
    for i, f in enumerate(filters):
        k = 7 if i == 0 else (5 if i < 3 else 3)
        name = "conv4_gradcam" if i == len(filters) - 1 else f"conv{i+1}"
        x = layers.Conv1D(f, kernel_size=k, padding="same", name=name)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling1D(pool_size=2)(x)
        x = layers.Dropout(dropout)(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)
    return keras.Model(inputs, outputs, name="cnn_1d")


def res_block(x, filters, kernel_size, dropout):
    shortcut = x
    x = layers.Conv1D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv1D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x)
    # Project shortcut if channel count differs
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    return x


def build_rescnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, dropout=0.3):
    inputs = keras.Input(shape=input_shape, name="ecg_input")
    x = layers.Conv1D(64, 7, padding="same")(inputs)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(2)(x)

    for filters in [64, 128, 256]:
        x = res_block(x, filters, 5, dropout)
        x = res_block(x, filters, 5, dropout)
        x = layers.MaxPooling1D(2)(x)

    # Name last conv block for Grad-CAM
    x = layers.Conv1D(256, 3, padding="same", name="res_final_conv")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)
    return keras.Model(inputs, outputs, name="rescnn_1d")


def build_lstm(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, dropout=0.3):
    inputs = keras.Input(shape=input_shape, name="ecg_input")
    x = layers.Conv1D(64, 7, padding="same")(inputs)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=dropout))(x)
    x = layers.Bidirectional(layers.LSTM(64, dropout=dropout))(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)
    return keras.Model(inputs, outputs, name="bilstm")


# Quick summary
for build_fn in [build_cnn, build_rescnn, build_lstm]:
    m = build_fn()
    print(f"{m.name:12s}  params={m.count_params():>10,}")
    del m

## 6. Focal Loss

Binary cross-entropy treats every sample equally. Focal loss down-weights
easy examples (where the model is already confident) and focuses gradient
on hard examples — particularly useful for rare classes like HYP (~2% of PTB-XL).

$\text{FL}(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$

We use α=0.25, γ=2.0 (the Lin et al. defaults from the original paper).

In [ ]:
def binary_focal_loss(alpha=0.25, gamma=2.0):
    """Multi-label focal loss with sigmoid outputs."""
    def loss_fn(y_true, y_pred):
        eps = keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1.0 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1.0 - alpha)
        return -tf.reduce_mean(alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt))
    loss_fn.__name__ = "binary_focal_loss"
    return loss_fn

## 7. Keras Tuner — hyperparameter search on the CNN

We tune the CNN (cheapest to train) over the hyperparameters most likely
to affect performance:

| Hyperparam      | Range                  |
|-----------------|------------------------|
| filters\_mult   | [0.5, 1.0, 1.5, 2.0]   → scales (32, 64, 128, 256) |
| dropout         | [0.2, 0.3, 0.4, 0.5]   |
| dense\_units    | [64, 128, 256]         |
| learning\_rate  | log-uniform [1e-4, 3e-3] |
| loss            | {BCE, focal}           |

**Hyperband** is used because it kills bad trials early — much faster than
random search. `max_epochs=20` per trial, `factor=3`.

In [ ]:
TUNE = True        # Set to False to skip tuning and use defaults
TUNER_MAX_EPOCHS = 20
TUNER_DIR = "keras_tuner"

def build_tunable_cnn(hp):
    mult = hp.Choice("filters_mult", [0.5, 1.0, 1.5, 2.0])
    dropout = hp.Float("dropout", 0.2, 0.5, step=0.1)
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    lr = hp.Float("lr", 1e-4, 3e-3, sampling="log")
    loss_type = hp.Choice("loss", ["bce", "focal"])

    base_filters = (32, 64, 128, 256)
    filters = tuple(max(8, int(f * mult)) for f in base_filters)

    model = build_cnn(filters=filters, dropout=dropout, dense_units=dense_units)
    loss = binary_focal_loss() if loss_type == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
    )
    return model


if TUNE:
    tuner = kt.Hyperband(
        build_tunable_cnn,
        objective=kt.Objective("val_auc", direction="max"),
        max_epochs=TUNER_MAX_EPOCHS,
        factor=3,
        directory=TUNER_DIR,
        project_name="ecg_cnn",
        overwrite=False,
        seed=SEED,
    )
    tuner.search_space_summary()

In [ ]:
if TUNE:
    tuner.search(
        train_ds,
        validation_data=val_ds,
        epochs=TUNER_MAX_EPOCHS,
        callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                           patience=4, restore_best_weights=True)],
        verbose=1,
    )
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_config = {k: best_hps.get(k) for k in
                   ["filters_mult", "dropout", "dense_units", "lr", "loss"]}
    print("Best hyperparameters:", best_config)

    with open(os.path.join(ART_DIR, "tuner_results.json"), "w") as f:
        json.dump(best_config, f, indent=2)
else:
    best_config = {"filters_mult": 1.0, "dropout": 0.3,
                   "dense_units": 128, "lr": 1e-3, "loss": "focal"}
    print("Tuning skipped. Using defaults:", best_config)

## 8. Final training runs

In [ ]:
# MLflow setup — optional, safe to skip if not installed
try:
    import mlflow
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("ECG_PTBXL_Final")
    USE_MLFLOW = True
except Exception as e:
    print(f"MLflow not available ({e}); continuing without tracking.")
    USE_MLFLOW = False


def train_model(model, name, loss_name="focal", lr=1e-3,
                epochs=50, use_augmentation=True):
    loss_fn = binary_focal_loss() if loss_name == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss_fn,
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc", multi_label=True),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    ckpt_path = os.path.join(ART_DIR, f"ecg_{name}_final.keras")
    cb = [
        callbacks.ModelCheckpoint(ckpt_path, monitor="val_auc", mode="max",
                                  save_best_only=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                    patience=5, min_lr=1e-6, verbose=1),
        callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                patience=10, restore_best_weights=True, verbose=1),
    ]

    tr_ds = train_ds if use_augmentation else make_dataset(
        X_train, y_train, BATCH_SIZE, augment=False, shuffle=True)

    if USE_MLFLOW:
        with mlflow.start_run(run_name=name):
            mlflow.log_params({"loss": loss_name, "lr": lr, "epochs": epochs,
                               "augmentation": use_augmentation,
                               "n_params": model.count_params()})
            history = model.fit(tr_ds, validation_data=val_ds,
                                epochs=epochs, callbacks=cb, verbose=2)
            for k, v in history.history.items():
                for i, val in enumerate(v):
                    mlflow.log_metric(k, val, step=i)
    else:
        history = model.fit(tr_ds, validation_data=val_ds,
                            epochs=epochs, callbacks=cb, verbose=2)
    return history

In [ ]:
# Build final CNN using the tuned hyperparameters
mult = best_config["filters_mult"]
tuned_filters = tuple(max(8, int(f * mult)) for f in (32, 64, 128, 256))
cnn_model = build_cnn(
    filters=tuned_filters,
    dropout=best_config["dropout"],
    dense_units=best_config["dense_units"],
)
cnn_history = train_model(cnn_model, "cnn",
                          loss_name=best_config["loss"],
                          lr=best_config["lr"],
                          epochs=50)

In [ ]:
# Residual CNN with default config
rescnn_model = build_rescnn(dropout=0.3)
rescnn_history = train_model(rescnn_model, "rescnn",
                             loss_name="focal", lr=1e-3, epochs=50)

In [ ]:
# BiLSTM with default config
lstm_model = build_lstm(dropout=0.3)
lstm_history = train_model(lstm_model, "lstm",
                           loss_name="focal", lr=1e-3, epochs=40)

## 9. Save histories for the comparison notebook

In [ ]:
histories = {
    "cnn":    cnn_history.history,
    "rescnn": rescnn_history.history,
    "lstm":   lstm_history.history,
}
with open(os.path.join(ART_DIR, "histories.pkl"), "wb") as f:
    pickle.dump(histories, f)

print("Training complete. Artifacts saved:")
for f in sorted(os.listdir(ART_DIR)):
    size_mb = os.path.getsize(os.path.join(ART_DIR, f)) / 1e6
    print(f"  {f:40s} {size_mb:>8.2f} MB")

---

**Next step:** open `comparison.ipynb` for full evaluation and model comparison.